# Exercise 2 · Pull the course's new files, even after you changed one

You will pull a new day into your clone, and recover when `git pull` stops on a file you changed.

**Goal:** Get the course team's newest commit into your clone with `git pull`, and keep your own change to a notebook.

**Why it matters:** The course team adds and fixes days during the week. `git pull` is how the new files reach your computer.

**Before you start:**
- You cloned the course (course README, "Set up once", step 4).
- A clone is your copy of a repository. A commit is one saved change in its history.
- Nothing here goes online. A folder on your computer stands in for GitHub.

**What the check measures:** your clone has the course team's newest commit and its new file, and your saved copy holds your change.

**Done when:** the last cell prints `Done:`.

## The situation

`practice/course.git` stands in for the course on GitHub. `practice/my-course` is your clone of it. You type an answer into day 1. Then the course team adds day 2 and fixes day 1. The files are made up.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It builds `practice/` as it is after Step 6: the course team's commit pulled into `practice/my-course`, and a trainee's answer saved in `practice/my-day-01.ipynb` (simulated).

Expect `Wrote practice/` lines, or `Kept` for a folder that is already there.

In [ ]:
# Step 1: build practice/course.git (a stand-in for GitHub) and practice/my-course (your clone of it)
import json
import subprocess
import tempfile
from pathlib import Path

PRACTICE = Path("practice")
COURSE = PRACTICE / "course.git"
MINE = PRACTICE / "my-course"
TEAM = ["-c", "user.name=Course team", "-c", "user.email=team@example.invalid"]


def notebook(title: str) -> str:
    cells = [{"cell_type": "markdown", "id": "title", "metadata": {}, "source": [title]},
             {"cell_type": "code", "execution_count": None, "id": "answer", "metadata": {}, "outputs": [],
              "source": ["# Your answer here\n"]}]
    return json.dumps({"cells": cells, "metadata": {}, "nbformat": 4, "nbformat_minor": 5}, indent=1) + "\n"


def git(*args, cwd=None) -> str:
    return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True).stdout


def write(folder, files: dict) -> None:
    for name, text in files.items():
        (Path(folder) / name).parent.mkdir(parents=True, exist_ok=True)
        (Path(folder) / name).write_text(text, encoding="utf-8")


PRACTICE.mkdir(exist_ok=True)
if COURSE.exists():
    print("Kept practice/course.git: it is already there.")
else:
    git("init", "-q", "--bare", "-b", "main", str(COURSE))
    with tempfile.TemporaryDirectory() as team:
        git("clone", "-q", str(COURSE.resolve()), team)
        write(team, {"README.md": "# Course\n\nOne folder per day.\n", "day-01/day-01.ipynb": notebook("# Day 1")})
        git(*TEAM, "add", "-A", cwd=team)
        git(*TEAM, "commit", "-qm", "Day 1", cwd=team)
        git("push", "-q", "origin", "HEAD:main", cwd=team)
    print("Wrote practice/course.git: the course, with one commit")
if MINE.exists():
    print("Kept practice/my-course: it is already there.")
else:
    git("clone", "-q", str(COURSE.resolve()), str(MINE))
    print("Wrote practice/my-course: your clone of the course")

# Steps 3 to 6, as a trainee does them (simulated): change day 1, the team pushes, pull stops,
# save a copy, restore, pull again.
SAVED = PRACTICE / "my-day-01.ipynb"
if SAVED.exists():
    print("Kept practice/my-day-01.ipynb: it is already there.")
else:
    answer = notebook("# Day 1").replace("# Your answer here", "answer = 42  # a trainee's answer")
    (MINE / "day-01" / "day-01.ipynb").write_text(answer, encoding="utf-8")
    with tempfile.TemporaryDirectory() as team:
        git("clone", "-q", str(COURSE.resolve()), team)
        write(team, {"day-02/day-02.ipynb": notebook("# Day 2"), "day-01/day-01.ipynb": notebook("# Day 1, fixed")})
        git(*TEAM, "add", "-A", cwd=team)
        git(*TEAM, "commit", "-qm", "Add day 2 and fix day 1", cwd=team)
        git("push", "-q", "origin", "HEAD:main", cwd=team)
    stopped = subprocess.run(["git", "pull", "-q"], cwd=MINE, capture_output=True, text=True)
    print("git pull stopped:", stopped.returncode != 0)
    SAVED.write_text((MINE / "day-01" / "day-01.ipynb").read_text(encoding="utf-8"), encoding="utf-8")
    git("restore", "day-01/day-01.ipynb", cwd=MINE)
    git("pull", "-q", cwd=MINE)
    print("Wrote practice/my-day-01.ipynb, and pulled the course team's commit into practice/my-course")

## Step 2 · Open a terminal in your clone

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-02/exercise-02-pull-the-new-files/practice/my-course
git log --oneline
```

Expect one line: a short ID, then `Day 1`.

## Step 3 · Change day 1, as you do in class

```sh
code day-01/day-01.ipynb
```

Or open the file from VS Code's Explorer. Type any answer in its code cell. Save it (Ctrl+S or Cmd+S). Then:

```sh
git status
```

Expect `modified:   day-01/day-01.ipynb`.

## Step 4 · The course team pushes a new commit

Run the cell. It plays the course team: it adds `day-02/day-02.ipynb`, fixes day 1, and pushes to `practice/course.git`.

Expect `The course team pushed one commit`.

In [ ]:
# Step 4: the course team adds day 2, fixes day 1, and pushes to practice/course.git
MESSAGE = "Add day 2 and fix day 1"

if "COURSE" not in globals():
    print("Run the Step 1 cell first, then this cell.")
elif MESSAGE in git("--git-dir", str(COURSE), "log", "--format=%s", "main").splitlines():
    print("Kept: the course team's commit is already in practice/course.git.")
else:
    with tempfile.TemporaryDirectory() as team:
        git("clone", "-q", str(COURSE.resolve()), team)
        write(team, {"day-02/day-02.ipynb": notebook("# Day 2"), "day-01/day-01.ipynb": notebook("# Day 1, fixed")})
        git(*TEAM, "add", "-A", cwd=team)
        git(*TEAM, "commit", "-qm", MESSAGE, cwd=team)
        git("push", "-q", "origin", "HEAD:main", cwd=team)
    print(f"The course team pushed one commit to practice/course.git: {MESSAGE}")

## Step 5 · Pull, and read why it stops

```sh
git pull
```

Expect it to stop:

```text
error: Your local changes to the following files would be overwritten by merge:
	day-01/day-01.ipynb
Please commit your changes or stash them before you merge.
Aborting
```

Git stops because the course team changed the same file you changed. Nothing is lost yet.

## Step 6 · Save your copy, undo your change, pull again

This is the recovery in the course README, "Get today's files":

> If `git pull` stops and names a notebook you changed, save your copy under a new name, undo your change, and pull again.

Use the path that `git pull` named:

```sh
cp day-01/day-01.ipynb ../my-day-01.ipynb
git restore day-01/day-01.ipynb
git pull
git log --oneline
```

Expect `Fast-forward` and `2 files changed`. Then two commits, newest first: `Add day 2 and fix day 1`, then `Day 1`. Your answer is safe in `practice/my-day-01.ipynb`.

<details><summary>Show the answer: why copy before git restore?</summary>

`git restore` writes the committed file over yours ([git-restore](https://git-scm.com/docs/git-restore)). Why copy first: the copy is the only place your answer stays.

</details>

## Step 7 · Do it in your real course folder

Open a new terminal. It starts in the course folder. Run:

```sh
git pull
uv sync
```

If `git pull` stops and names a file, use the first three lines of Step 6 with that path. Save the copy outside the course folder, for example in `~/`. The check does not cover this step.

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It deletes `practice/course.git`, `practice/my-course` and `practice/my-day-01.ipynb`. Then set it back to `False`. Run the Step 1 cell. Then go to Step 2 in a new terminal.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, delete the practice folders and your saved copy
import shutil
from pathlib import Path

START_OVER = False

if not START_OVER:
    print("Nothing changed.")
else:
    for folder in (Path("practice/course.git"), Path("practice/my-course")):
        shutil.rmtree(folder, ignore_errors=True)
    Path("practice/my-day-01.ipynb").unlink(missing_ok=True)
    print("Deleted the practice files. Set START_OVER back to False, then run the Step 1 cell.")

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: compare your clone with practice/course.git, and read your saved copy
import subprocess
from pathlib import Path

PRACTICE = Path("practice")
COURSE, MINE, SAVED = PRACTICE / "course.git", PRACTICE / "my-course", PRACTICE / "my-day-01.ipynb"


def read_git(*args, strip=True) -> str:
    """Return git's output, or "" if the command fails."""
    done = subprocess.run(["git", *args], capture_output=True, text=True)
    return "" if done.returncode else done.stdout.strip() if strip else done.stdout


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    if not (COURSE.is_dir() and (MINE / ".git").is_dir()):
        return "practice/course.git or practice/my-course is missing. Run the Step 1 cell."
    course = ["--git-dir", str(COURSE)]
    if "Add day 2 and fix day 1" not in read_git(*course, "log", "--format=%s", "main").splitlines():
        return "the course team has not pushed yet. Run the Step 4 cell."
    if read_git("-C", str(MINE), "rev-parse", "HEAD") != read_git(*course, "rev-parse", "main"):
        return "practice/my-course does not have the course team's newest commit. Run git pull there (Step 5 and Step 6)."
    print("ok: your clone has the course team's newest commit")
    if not (MINE / "day-02" / "day-02.ipynb").is_file():
        return "practice/my-course/day-02/day-02.ipynb is missing. Run git pull in practice/my-course (Step 6)."
    print("ok: day-02/day-02.ipynb arrived")
    if not SAVED.is_file():
        return "practice/my-day-01.ipynb is missing. Save your copy before git restore (Step 6)."
    first = read_git(*course, "rev-list", "--max-parents=0", "main")
    versions = {read_git(*course, "show", f"{commit}:day-01/day-01.ipynb", strip=False) for commit in (first, "main")}
    if SAVED.read_text(encoding="utf-8") in versions:
        return "practice/my-day-01.ipynb is the course's file, not yours. Type an answer in day 1 (Step 3), then save it before git restore (Step 6)."
    print("ok: practice/my-day-01.ipynb holds your change")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the Step 1 cell, then this cell."
print(f"Not yet: {problem}" if problem else
      "Done: you pulled the course team's new files and kept your own change.")

## What you learned

- `git pull` brings the course team's new commits into your clone.
- When it stops on a file you changed: copy the file aside, `git restore` it, and pull again.

**Reflect:** Which notebook in your real course folder have you changed? Where will you save your copy of it?